# THROWAWAY test: Kyoto-basket QEXTRA subtraction vs. direct construction

Not part of the numbered pipeline - exploratory only, safe to delete once done.

Basket decision is now settled: the non-CO2-GHG basket is the **Kyoto gases** (CO2+CH4+N2O+F-Gases),
**no Montreal Protocol halogens**, and the earlier "expanded" basket (+ tropospheric ozone +
CH4-oxidation stratospheric H2O) is dropped.

Remaining open question: given that basket, should the non-CO2-GHG contribution be computed by
**subtraction** (QEXTRA Kyoto-basket run minus QEXTRA CO2-only run) or by **direct construction**
(a QEXTRA run driven only by CH4+N2O+F-Gases ERF, no CO2 at all)? This notebook runs both for the
same 2 scenarios used in the earlier basket-definition checks:

1. `ANTHROPOGENIC` (native) - harvests CO2/CH4/N2O/F-Gases component ERFs for passes 4-6
2. `GHG` (native, MAGICC's built-in Kyoto+Montreal aggregation) - kept for reference only
3. `CO2` (native, standalone)
4. QEXTRA: CO2-only (harvested from pass 1)
5. QEXTRA: Kyoto basket, CO2+CH4+N2O+F-Gases (harvested from pass 1)
6. QEXTRA: Kyoto basket **without CO2**, CH4+N2O+F-Gases only (harvested from pass 1) - direct construction

Sensitivity test: compare `(5 - 4)` [subtraction] against `6` [direct construction], per-member
delta first, then median.

Uses raw (unrebased) GSAT throughout - see notes/ in this repo for why the AR6-assessment-calibrated GSAT is wrong for these comparisons.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("pint.util").setLevel(logging.ERROR)

import os
import platform
import time
from pathlib import Path

import dotenv
import pandas as pd
import pandas_indexing as pix
from pandas_openscm.io import load_timeseries_csv

from gcages.cmip7_scenariomip.scm_running import CMIP7ScenarioMIPSCMRunner

from utils import sanitize_label, write_magicc_extra_rf_file

dotenv.load_dotenv()
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

In [2]:
ENSEMBLE_MEMBERS = 600
N_PROCESSES = 8

SCENARIOS = [
    ("TIAM-ECN 1.1", "ENGAGE-INDCi2030-900_NZCO2"),
    ("WITCH 5.0", "ENGAGE-INDCi2030-1200-NDCp_NZCO2"),
]
FILE = OUTPUT_FOLDER / "101_netzero_co2_extended.csv"

BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"
MAGICC_EXE_PATH = BASE_DIR / "magicc-v7.6.0a3" / "bin"
MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE = BASE_DIR / "magicc-v7.6.0a3" / "configs" / "magicc-ar7-fast-track-drawnset-v0-3-0.json"

if platform.system() == "Darwin" and platform.processor() == "arm":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc-darwin-arm64"
    os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/Cellar/gcc/15.2.0_1/lib/gcc/current"

start = load_timeseries_csv(
    FILE, index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int, out_columns_name="year",
)
netzero_timings = pd.read_csv(OUTPUT_FOLDER / "101_netzero_timings.csv", index_col=[0, 1])

In [3]:
CO2_ONLY = ("Effective Radiative Forcing|CO2",)

KYOTO_ONLY = (
    "Effective Radiative Forcing|CO2",
    "Effective Radiative Forcing|CH4",
    "Effective Radiative Forcing|N2O",
    "Effective Radiative Forcing|F-Gases",
)

NON_CO2_KYOTO_ONLY = (
    "Effective Radiative Forcing|CH4",
    "Effective Radiative Forcing|N2O",
    "Effective Radiative Forcing|F-Gases",
)

ALL_HARVEST_COMPONENTS = tuple(sorted(set(KYOTO_ONLY)))

In [4]:
def build_scm_runner(output_variables):
    runner = CMIP7ScenarioMIPSCMRunner.from_cmip7_scenariomip_config(
        n_processes=N_PROCESSES,
        magicc_exe_path=MAGICC_EXE,
        magicc_prob_distribution_path=MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE,
        historical_emissions_path=CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS,
        output_variables=output_variables,
        batch_size_scenarios=15,
    )
    runner.climate_models_cfgs["MAGICC7"] = runner.climate_models_cfgs["MAGICC7"][:ENSEMBLE_MEMBERS]
    return runner


def raw_gsat(scm_results):
    """Raw simulated GSAT, no assessed-calibration rebasing."""
    raw = scm_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]
    raw = raw.reset_index().set_index("run_id")
    year_columns = [c for c in raw.columns if isinstance(c, int)]
    return raw[year_columns]


def run_native(scenario_to_run, rf_total_runmodus, output_variables=("Surface Air Temperature Change",)):
    runner = build_scm_runner(output_variables)
    runner.climate_models_cfgs["MAGICC7"] = [
        {**c, "rf_total_runmodus": rf_total_runmodus} for c in runner.climate_models_cfgs["MAGICC7"]
    ]
    return runner(scenario_to_run)


def sum_components(erf_results, components):
    summed = erf_results.loc[pix.isin(variable=list(components))].groupby(level="run_id").sum(numeric_only=True)
    year_columns = [c for c in summed.columns if isinstance(c, int)]
    return summed[year_columns]


def run_qextra(scenario_to_run, forcing_sum_df, tag):
    qdir = Path("/tmp") / "zz_ghg_basket_test" / tag
    for run_id, row in forcing_sum_df.iterrows():
        write_magicc_extra_rf_file(
            qdir / f"member_{run_id}.IN", years=list(row.index), values=list(row.values),
            description=f"{tag} member {run_id}",
        )
    runner = build_scm_runner(("Surface Air Temperature Change",))
    cfgs = runner.climate_models_cfgs["MAGICC7"]
    runner.climate_models_cfgs["MAGICC7"] = [
        {**c, "file_extra_rf": str(qdir / f"member_{run_id}.IN"), "rf_extra_read": 1, "rf_total_runmodus": "QEXTRA"}
        for run_id, c in zip(forcing_sum_df.index, cfgs)
    ]
    return raw_gsat(runner(scenario_to_run))

In [5]:
RAW_GSAT_DIR = Path("/tmp/zz_ghg_basket_test/raw_gsat")
RAW_GSAT_DIR.mkdir(parents=True, exist_ok=True)

results_rows = []
gsat_by_run = {}

for model, scenario in SCENARIOS:
    tag = sanitize_label(f"{model}_{scenario}")
    print(f"\n=== {model} | {scenario} ===")
    scenario_to_run = start.loc[pd.IndexSlice[model, scenario, :, :, :], :]
    base_scenario = scenario.replace("_NZCO2", "")
    nz_year = int(netzero_timings.loc[(model, base_scenario), "year_netzero_co2"])

    t0 = time.time()
    print("1/6 ANTHROPOGENIC (harvest)...")
    anthro_results = run_native(scenario_to_run, "ANTHROPOGENIC", ("Surface Air Temperature Change", *ALL_HARVEST_COMPONENTS))
    gsat_anthro = raw_gsat(anthro_results)
    print(f"  done in {time.time()-t0:.0f}s")

    t0 = time.time()
    print("2/6 native GHG (reference only, includes Montreal)...")
    gsat_native_ghg = raw_gsat(run_native(scenario_to_run, "GHG"))
    print(f"  done in {time.time()-t0:.0f}s")

    t0 = time.time()
    print("3/6 native CO2...")
    gsat_native_co2 = raw_gsat(run_native(scenario_to_run, "CO2"))
    print(f"  done in {time.time()-t0:.0f}s")

    co2_sum = sum_components(anthro_results, CO2_ONLY)
    kyoto_sum = sum_components(anthro_results, KYOTO_ONLY)
    non_co2_kyoto_sum = sum_components(anthro_results, NON_CO2_KYOTO_ONLY)

    t0 = time.time()
    print("4/6 QEXTRA CO2-only...")
    gsat_qextra_co2 = run_qextra(scenario_to_run, co2_sum, f"{tag}_co2")
    print(f"  done in {time.time()-t0:.0f}s")

    t0 = time.time()
    print("5/6 QEXTRA Kyoto basket (CO2+CH4+N2O+F-Gases)...")
    gsat_qextra_kyoto = run_qextra(scenario_to_run, kyoto_sum, f"{tag}_kyoto")
    print(f"  done in {time.time()-t0:.0f}s")

    t0 = time.time()
    print("6/6 QEXTRA non-CO2 Kyoto basket (CH4+N2O+F-Gases, direct construction)...")
    gsat_qextra_non_co2_kyoto = run_qextra(scenario_to_run, non_co2_kyoto_sum, f"{tag}_nonco2kyoto")
    print(f"  done in {time.time()-t0:.0f}s")

    runs = {
        "1_ANTHROPOGENIC": gsat_anthro,
        "2_native_GHG": gsat_native_ghg,
        "3_native_CO2": gsat_native_co2,
        "4_QEXTRA_CO2only": gsat_qextra_co2,
        "5_QEXTRA_Kyoto": gsat_qextra_kyoto,
        "6_QEXTRA_NonCO2Kyoto": gsat_qextra_non_co2_kyoto,
    }
    gsat_by_run[(model, scenario)] = runs

    # Persist raw per-member GSAT so follow-up comparisons don't require rerunning MAGICC
    for run_label, gsat_df in runs.items():
        gsat_df.to_csv(RAW_GSAT_DIR / f"{tag}_{run_label}.csv")

    for run_label, gsat_df in runs.items():
        for year_label, year in [("nzco2_year", nz_year), ("2100", 2100)]:
            if year not in gsat_df.columns:
                continue
            vals = gsat_df[year]
            results_rows.append({
                "model": model, "scenario": scenario, "run": run_label, "year_label": year_label, "year": year,
                "median": vals.median(), "mean": vals.mean(), "p05": vals.quantile(0.05), "p95": vals.quantile(0.95),
            })

summary = pd.DataFrame(results_rows)
summary.to_csv("/tmp/zz_ghg_basket_test_summary.csv", index=False)
print("\nDone.")


=== TIAM-ECN 1.1 | ENGAGE-INDCi2030-900_NZCO2 ===
1/6 ANTHROPOGENIC (harvest)...


Climate models:   0%|          | 0/1 [00:00<?, ?it/s]

Scenario batch:   0%|          | 0/1 [00:00<?, ?it/s]

Climate models:   0%|          | 0.00/1.00 [00:00<?, ?it/s]

[WARNING] 15:35:45 - openscm_runner.adapters.magicc7.magicc7: Historical data has not been checked


...

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


Parallel runs:   0%|          | 0.00/596 [00:00<?, ?it/s]

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, whi

  done in 45s



Done.


In [6]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

for (model, scenario), runs in gsat_by_run.items():
    print(f"\n=== {model} | {scenario} ===")
    base_scenario = scenario.replace("_NZCO2", "")
    nz_year = int(netzero_timings.loc[(model, base_scenario), "year_netzero_co2"])

    anthro_2100 = runs["1_ANTHROPOGENIC"][2100]

    table_rows = []
    for run_label, gsat_df in runs.items():
        v2100 = gsat_df[2100]
        vnz = gsat_df[nz_year] if nz_year in gsat_df.columns else pd.Series(dtype=float)
        table_rows.append({
            "run": run_label,
            f"GSAT@{nz_year}(median)": vnz.median() if not vnz.empty else float("nan"),
            "GSAT@2100(median)": v2100.median(),
            "delta(nz->2100, median)": (v2100 - vnz).median() if not vnz.empty else float("nan"),
            "share of ANTHROPOGENIC 2100 GSAT": v2100.median() / anthro_2100.median(),
        })
    print(pd.DataFrame(table_rows).to_string(index=False))

    # "Non-GHG residual" implied by ANTHROPOGENIC minus each GHG-basket candidate
    print("\n  Implied non-GHG residual (ANTHROPOGENIC - GHG candidate), median GSAT@2100:")
    for run_label in ["2_native_GHG", "5_QEXTRA_Kyoto"]:
        residual = (runs["1_ANTHROPOGENIC"][2100] - runs[run_label][2100]).median()
        print(f"    ANTHROPOGENIC - {run_label}: {residual:.4f} degC")


=== TIAM-ECN 1.1 | ENGAGE-INDCi2030-900_NZCO2 ===
                 run  GSAT@2080(median)  GSAT@2100(median)  delta(nz->2100, median)  share of ANTHROPOGENIC 2100 GSAT
     1_ANTHROPOGENIC           1.683673           1.697551                 0.022700                          1.000000
        2_native_GHG           2.093247           2.071508                -0.020845                          1.220292
        3_native_CO2           1.406593           1.370445                -0.022918                          0.807307
    4_QEXTRA_CO2only           1.443104           1.420639                -0.017038                          0.836875
      5_QEXTRA_Kyoto           1.924896           1.919045                -0.004015                          1.130478
6_QEXTRA_NonCO2Kyoto           0.417771           0.427795                 0.010268                          0.252007

  Implied non-GHG residual (ANTHROPOGENIC - GHG candidate), median GSAT@2100:
    ANTHROPOGENIC - 2_native_GHG: -0.3566 de

In [7]:
print("\n\n=== METHODOLOGICAL COMPARISON: native subtraction vs QEXTRA subtraction (Kyoto basket) ===")
print("(all deltas computed per-member first, then median taken - not median-then-subtract)\n")

for (model, scenario), runs in gsat_by_run.items():
    print(f"\n=== {model} | {scenario} ===")
    base_scenario = scenario.replace("_NZCO2", "")
    nz_year = int(netzero_timings.loc[(model, base_scenario), "year_netzero_co2"])

    delta_native = runs["2_native_GHG"] - runs["3_native_CO2"]
    delta_qextra_kyoto = runs["5_QEXTRA_Kyoto"] - runs["4_QEXTRA_CO2only"]

    rows = []
    for label, delta_df in [
        ("native_GHG - native_CO2 (includes Montreal)", delta_native),
        ("QEXTRA_Kyoto - QEXTRA_CO2only (subtraction)", delta_qextra_kyoto),
    ]:
        d_nz = delta_df[nz_year] if nz_year in delta_df.columns else pd.Series(dtype=float)
        d_2100 = delta_df[2100]
        rows.append({
            "comparison (non-CO2-GHG delta)": label,
            f"delta@{nz_year}(median)": d_nz.median() if not d_nz.empty else float("nan"),
            "delta@2100(median)": d_2100.median(),
            "change_in_delta(nz->2100, median)": (d_2100 - d_nz).median() if not d_nz.empty else float("nan"),
        })
    print(pd.DataFrame(rows).to_string(index=False))


print("\n\n=== SENSITIVITY TEST: QEXTRA subtraction vs. direct construction (Kyoto basket, no Montreal) ===")
print("subtraction  = QEXTRA_Kyoto (CO2+CH4+N2O+F-Gases) - QEXTRA_CO2only")
print("direct       = QEXTRA_NonCO2Kyoto (CH4+N2O+F-Gases only, no CO2 in the run at all)")
print("(all deltas computed per-member first, then median taken - not median-then-subtract)\n")

for (model, scenario), runs in gsat_by_run.items():
    print(f"\n=== {model} | {scenario} ===")
    base_scenario = scenario.replace("_NZCO2", "")
    nz_year = int(netzero_timings.loc[(model, base_scenario), "year_netzero_co2"])

    subtraction = runs["5_QEXTRA_Kyoto"] - runs["4_QEXTRA_CO2only"]
    direct = runs["6_QEXTRA_NonCO2Kyoto"]
    # per-member difference between the two approaches themselves
    approach_gap = subtraction - direct

    rows = []
    for label, df in [
        ("subtraction (5 - 4)", subtraction),
        ("direct construction (6)", direct),
        ("gap: subtraction - direct", approach_gap),
    ]:
        d_nz = df[nz_year] if nz_year in df.columns else pd.Series(dtype=float)
        d_2100 = df[2100]
        rows.append({
            "quantity": label,
            f"value@{nz_year}(median)": d_nz.median() if not d_nz.empty else float("nan"),
            "value@2100(median)": d_2100.median(),
        })
    print(pd.DataFrame(rows).to_string(index=False))

    gap_2100 = approach_gap[2100]
    subtraction_2100 = subtraction[2100]
    print(f"\n  gap as % of subtraction@2100 (median of per-member ratio): "
          f"{(gap_2100 / subtraction_2100).median() * 100:.1f}%")
    print(f"  gap as % of subtraction@2100 (ratio of medians): "
          f"{gap_2100.median() / subtraction_2100.median() * 100:.1f}%")



=== METHODOLOGICAL COMPARISON: native subtraction vs QEXTRA subtraction (Kyoto basket) ===
(all deltas computed per-member first, then median taken - not median-then-subtract)


=== TIAM-ECN 1.1 | ENGAGE-INDCi2030-900_NZCO2 ===
             comparison (non-CO2-GHG delta)  delta@2080(median)  delta@2100(median)  change_in_delta(nz->2100, median)
native_GHG - native_CO2 (includes Montreal)            0.668933            0.670599                           0.003357
QEXTRA_Kyoto - QEXTRA_CO2only (subtraction)            0.459625            0.473135                           0.013260

=== WITCH 5.0 | ENGAGE-INDCi2030-1200-NDCp_NZCO2 ===
             comparison (non-CO2-GHG delta)  delta@2085(median)  delta@2100(median)  change_in_delta(nz->2100, median)
native_GHG - native_CO2 (includes Montreal)            0.570091            0.556032                          -0.015373
QEXTRA_Kyoto - QEXTRA_CO2only (subtraction)            0.377958            0.372457                          -0.005749


